# 🚀 ToolMovieAI - Colab GPU Acceleration Worker

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuocDung21/ToolMovieAI/blob/main/colab_gpu_worker.ipynb)

Notebook này biến Google Colab thành **Cụm tính toán GPU đám mây (Cloud AI Worker)** cho ToolMovieAI trên Mac M1/M2/M3 hoặc PC của bạn.

### ⚡ Tính năng tăng tốc:
1. **Demucs v4 (htdemucs)**: Tách nhạc nền (BGM) và giọng nói (Vocal) sạch bóng chuẩn phòng thu bằng GPU CUDA.
2. **Faster-Whisper Large-v3**: Nhận diện lời thoại tiếng Trung & đa ngôn ngữ siêu tốc độ (15x - 20x realtime) với độ chính xác cao nhất.
3. **Zero-Config Cloudflare Tunnel**: Tạo đường truyền bảo mật kết nối trực tiếp về máy cá nhân của bạn mà **không cần đăng ký tài khoản** hay mở port modem!

### Bước 1: Kiểm tra GPU Nvidia của Google Colab
> *Lưu ý: Vào menu **Runtime** > **Change runtime type** > Chọn **T4 GPU** (hoặc A100/L4 nếu có Colab Pro).*

In [ ]:
!nvidia-smi

### Bước 2: Cài đặt các thư viện AI & Máy chủ API

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q fastapi uvicorn python-multipart faster-whisper demucs pycloudflared soundfile

### Bước 3: Khởi chạy Máy chủ GPU Worker & Cloudflare Tunnel
Nhấn nút ▶ để chạy cell bên dưới. Sau ~15 giây, bạn sẽ nhận được một đường link có dạng:
👉 `https://xxxxx.trycloudflare.com`

Hãy **copy đường link này** và dán vào mục **"⚡ Tăng tốc GPU Cloud (Colab)"** trên giao diện Web Studio máy Mac của bạn.

In [ ]:
import os
import subprocess
import threading
import time

# Tải file worker mới nhất từ GitHub repo ToolMovieAI nếu chưa có
if not os.path.exists("colab_worker.py"):
    !wget -q https://raw.githubusercontent.com/QuocDung21/ToolMovieAI/main/colab_worker.py

print("⏳ Đang khởi tạo đường hầm Cloudflare Tunnel...")
from pycloudflared import try_cloudflare

# Chạy uvicorn server trong background
def start_server():
    import uvicorn
    from colab_worker import app
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

t = threading.Thread(target=start_server, daemon=True)
t.start()
time.sleep(2)

public_url = try_cloudflare(port=8000)

print("\n" + "="*65)
print("🎉 MÁY CHỦ GPU COLAB ĐÃ SẴN SÀNG!")
print(f"👉 COPY LINK NÀY DÁN VÀO WEB STUDIO TRÊN MAC CỦA BẠN:")
print(f"   {public_url}")
print("="*65 + "\n")

# Giữ cell chạy để máy chủ không bị tắt
try:
    while True:
        time.sleep(10)
except KeyboardInterrupt:
    print("Máy chủ đã dừng.")